# .Stat Suite: publish and clean up

This walkthrough covers only the .Stat side. The FMR is only read: we
publish its structures to .Stat, create a difference, and clean .Stat
up so it matches the FMR again.

> **Before you start:** run step 3 of
> `../fmr/stat_fmr_sync.ipynb`, which
> uploads the structures to the FMR.

| # | Step |
|---|------|
| 1 | Setup |
| 2 | Prepare the structures |
| 3 | Publish FMR → .Stat |
| 4 | Create a difference |
| 5 | Compare FMR and .Stat |
| 6 | Deletion order |
| 7 | Clean up .Stat and verify |
| 8 | Recap and tear-down |

> ⚠️ Experimental pysdmx APIs: they may change.

## 1 · Setup

We install pysdmx and import what we need. Two classes do the work:

- `RestService` reads from the FMR and .Stat,
- `StatUploader` writes to and deletes from .Stat.

Both use **SDMX-ML 2.1**.

In [ ]:
%pip install --quiet ./pysdmx-1.24.0rc1-py3-none-any.whl[all]
%pip install certifi

import base64
import json
import os
import re
import time
from io import BytesIO
from pathlib import Path

import pandas as pd
from msgspec.structs import replace

from pysdmx.api.qb import (
    ApiVersion,
    RestService,
    StructureDetail,
    StructureFormat,
    StructureQuery,
    StructureReference,
    StructureType,
)
from pysdmx.api.stat import StatUploader
from pysdmx.errors import Invalid, NotFound
from pysdmx.io import read_sdmx
from pysdmx.io.format import Format
from pysdmx.model import (
    AgencyScheme,
    Codelist,
    Component,
    Components,
    ConceptScheme,
    Dataflow,
    DataStructureDefinition,
    Role,
)
from pysdmx.model.__base import DataType

### Configuration

Endpoints and credentials come from environment variables
(`FMR_ENDPOINT`, `FMR_USER` + `FMR_PASSWORD` or `FMR_TOKEN`, `NSI`,
`TRANSFER`, `SPACE`, `AGENCY`, default `BIS`).

The .Stat token is read from a file (`DOTSTAT_TOKEN_FILE`) on every
call, because it expires after 10 minutes.

In [2]:
FMR_ENDPOINT = os.environ.get(
    "FMR_ENDPOINT", "https://sdmxexpwsfa3.sdmx.solutions/fmr/sdmx/v2"
)
FMR_TOKEN = os.environ.get("FMR_TOKEN")
FMR_USER = os.environ.get("FMR_USER")
FMR_PASSWORD = os.environ.get("FMR_PASSWORD")
DOTSTAT_TOKEN_FILE = os.environ.get("DOTSTAT_TOKEN_FILE", "dotstat.token")

NSI = os.environ.get("NSI", "https://nsi-demo-stable.siscc.org")
TRANSFER = os.environ.get("TRANSFER", "https://transfer-demo.siscc.org/3")
SPACE = os.environ.get("SPACE", "staging:SIS-CC-stable")
AGENCY = os.environ.get("AGENCY", "BIS")

# Every artefact this demo creates starts with this prefix.
PREFIX = "PX_SYNC"
# Structures are read and submitted to .Stat as SDMX-ML 2.1.
STAT_FORMAT = Format.STRUCTURE_SDMX_ML_2_1

In [3]:
def dotstat_token() -> str:
    """Read the .Stat bearer token afresh (they live 10 minutes)."""
    raw = Path(DOTSTAT_TOKEN_FILE).read_text()
    return re.sub(r"[^A-Za-z0-9._-]", "", raw)


def minutes_left(token: str) -> float:
    """Minutes until a JWT expires (decodes the payload, no checks)."""
    payload = token.split(".")[1]
    payload += "=" * (-len(payload) % 4)
    claims = json.loads(base64.urlsafe_b64decode(payload))
    return (claims["exp"] - time.time()) / 60


def sdmx_service(base: str, token: str | None = None) -> RestService:
    """An SDMX-REST v2 reader for ``base``, in SDMX-ML 2.1."""
    return RestService(
        api_endpoint=base,
        api_version=ApiVersion.V2_0_0,
        structure_format=StructureFormat.SDMX_ML_2_1,
        token=token,
        timeout=120,
    )


def fmr_service() -> RestService:
    """An FMR reader, with HTTP Basic when FMR_USER is set.

    RestService only sends bearer tokens, so for an FMR behind HTTP
    Basic (like the workshop one) the header is set directly.
    """
    svc = sdmx_service(FMR_ENDPOINT, FMR_TOKEN)
    if FMR_USER and not FMR_TOKEN:
        pair = f"{FMR_USER}:{FMR_PASSWORD}".encode()
        svc._headers["Authorization"] = (
            f"Basic {base64.b64encode(pair).decode()}"
        )
    return svc


def stat_uploader() -> StatUploader:
    """A .Stat submission client built with the current token."""
    return StatUploader(
        nsi_endpoint=NSI,
        transfer_endpoint=TRANSFER,
        dataspace=SPACE,
        token=dotstat_token(),
    )


if Path(DOTSTAT_TOKEN_FILE).exists():
    print(f".Stat token valid for {minutes_left(dotstat_token()):.1f} min")
else:
    print(f"No .Stat token yet: save one to {DOTSTAT_TOKEN_FILE!r}")

No .Stat token yet: save one to 'dotstat.token'


## 2 · Prepare the structures

We rebuild the same ECB-based structures as in the main notebook. We
need them to create the difference and to reset .Stat.

In [4]:
ECB_STRUCTURES = (
    "https://data-api.ecb.europa.eu/service/"
    "dataflow/ECB/EXR/latest?references=descendants"
)
ecb = read_sdmx(ECB_STRUCTURES, validate=False)
pd.Series([type(s).__name__ for s in ecb.structures]).value_counts()

Codelist                   11
AgencyScheme                1
ConceptScheme               1
DataStructureDefinition     1
Dataflow                    1
Name: count, dtype: int64

In [5]:
def restamp(scheme):
    """The same item scheme, under AGENCY and the demo prefix."""
    new_id = f"{PREFIX}_{scheme.id}"
    old = f"={scheme.agency}:{scheme.id}({scheme.version})"
    new = f"={AGENCY}:{new_id}({scheme.version})"
    items = [
        replace(item, urn=item.urn.replace(old, new) if item.urn else None)
        for item in scheme.items
    ]
    return replace(scheme, agency=AGENCY, id=new_id, urn=None, items=items)


concepts = restamp(
    next(s for s in ecb.structures if isinstance(s, ConceptScheme))
)
codelists = {
    s.id: restamp(s) for s in ecb.structures if isinstance(s, Codelist)
}
schemes = [concepts, *codelists.values()]
pd.DataFrame(
    [(s.short_urn, len(s.items), s.name) for s in schemes],
    columns=["short URN", "items", "name"],
)

,short URN,items,name
0,ConceptScheme=BIS:PX_SYNC_ECB_CONCEPTS(1.0),507,ECB concepts
1,Codelist=BIS:PX_SYNC_CL_COLLECTION(1.0),10,Collection indicator code list
2,Codelist=BIS:PX_SYNC_CL_CURRENCY(1.0),369,Currency code list
3,Codelist=BIS:PX_SYNC_CL_DECIMALS(1.0),16,Decimals code list
4,Codelist=BIS:PX_SYNC_CL_EXR_SUFFIX(1.0),6,Exch. rate series variation code list
5,Codelist=BIS:PX_SYNC_CL_EXR_TYPE(1.0),36,Exchange rate type code list
6,Codelist=BIS:PX_SYNC_CL_FREQ(1.0),10,Frequency code list
7,Codelist=BIS:PX_SYNC_CL_OBS_CONF(1.0),11,Observation confidentiality code list
8,Codelist=BIS:PX_SYNC_CL_OBS_STATUS(1.0),19,Observation status code list
9,Codelist=BIS:PX_SYNC_CL_ORGANISATION(1.0),1044,Organisation code list


### Build two dataflows

On top of the ECB codelists we build two dataflows:

- `EXR`, which **stays** in the FMR;
- `EXR_OLD`, an older version with a `UNIT` attribute (`CL_UNIT`),
  which we will **retire**.

In [6]:
CONCEPT = {c.id: c for c in concepts.concepts}
URN = "urn:sdmx:org.sdmx.infomodel"

COMPONENTS = {
    # id: (role, codelist, data type, attachment level)
    "FREQ": (Role.DIMENSION, "CL_FREQ", DataType.STRING, None),
    "CURRENCY": (Role.DIMENSION, "CL_CURRENCY", DataType.STRING, None),
    "CURRENCY_DENOM": (Role.DIMENSION, "CL_CURRENCY", DataType.STRING, None),
    "EXR_TYPE": (Role.DIMENSION, "CL_EXR_TYPE", DataType.STRING, None),
    "EXR_SUFFIX": (Role.DIMENSION, "CL_EXR_SUFFIX", DataType.STRING, None),
    "TIME_PERIOD": (Role.DIMENSION, None, DataType.PERIOD, None),
    "OBS_VALUE": (Role.MEASURE, None, DataType.DOUBLE, None),
    "OBS_STATUS": (Role.ATTRIBUTE, "CL_OBS_STATUS", DataType.STRING, "O"),
    "UNIT": (Role.ATTRIBUTE, "CL_UNIT", DataType.STRING, "O"),
}


def build_flow(name: str, component_ids: list[str]) -> list:
    """A DSD and its dataflow, built from the re-stamped ECB schemes."""
    components = []
    for cid in component_ids:
        role, codelist, dtype, attachment = COMPONENTS[cid]
        codes = codelists[codelist] if codelist else None
        components.append(
            Component(
                id=cid,
                required=role != Role.ATTRIBUTE,
                role=role,
                concept=CONCEPT[cid],
                local_dtype=dtype,
                local_codes=codes,
                # the writers link the codelist through its full URN
                local_enum_ref=(
                    f"{URN}.codelist.{codes.short_urn}" if codes else None
                ),
                attachment_level=attachment,
            )
        )
    dsd = DataStructureDefinition(
        id=f"{PREFIX}_DSD_{name}",
        agency=AGENCY,
        version="1.0",
        name=f"Exchange rates ({name})",
        components=Components(components),
    )
    flow = Dataflow(
        id=f"{PREFIX}_DF_{name}",
        agency=AGENCY,
        version="1.0",
        name=f"Exchange rates ({name})",
        structure=f"{URN}.datastructure.{dsd.short_urn}",
    )
    return [dsd, flow]


EXR = [
    "FREQ",
    "CURRENCY",
    "CURRENCY_DENOM",
    "EXR_TYPE",
    "EXR_SUFFIX",
    "TIME_PERIOD",
    "OBS_VALUE",
    "OBS_STATUS",
]
keep = [
    concepts,
    *(
        codelists[c]
        for c in (
            "CL_FREQ",
            "CL_CURRENCY",
            "CL_EXR_TYPE",
            "CL_EXR_SUFFIX",
            "CL_OBS_STATUS",
        )
    ),
    *build_flow("EXR", EXR),
]
legacy = [codelists["CL_UNIT"], *build_flow("EXR_OLD", EXR + ["UNIT"])]

print("keep:  ", [s.short_urn for s in keep])
print("legacy:", [s.short_urn for s in legacy])

keep:   ['ConceptScheme=BIS:PX_SYNC_ECB_CONCEPTS(1.0)', 'Codelist=BIS:PX_SYNC_CL_FREQ(1.0)', 'Codelist=BIS:PX_SYNC_CL_CURRENCY(1.0)', 'Codelist=BIS:PX_SYNC_CL_EXR_TYPE(1.0)', 'Codelist=BIS:PX_SYNC_CL_EXR_SUFFIX(1.0)', 'Codelist=BIS:PX_SYNC_CL_OBS_STATUS(1.0)', 'DataStructure=BIS:PX_SYNC_DSD_EXR(1.0)', 'Dataflow=BIS:PX_SYNC_DF_EXR(1.0)']
legacy: ['Codelist=BIS:PX_SYNC_CL_UNIT(1.0)', 'DataStructure=BIS:PX_SYNC_DSD_EXR_OLD(1.0)', 'Dataflow=BIS:PX_SYNC_DF_EXR_OLD(1.0)']


### Start from a clean .Stat

.Stat rejects a structure that already exists, so we first remove
anything left over from a previous run.

In [ ]:
uploader = stat_uploader()
for artefact in [*reversed(legacy), *reversed(keep)]:
    try:
        uploader.delete_structure(artefact)
        print("removed  ", artefact.short_urn)
    except (Invalid, NotFound):  # noqa: PERF203
        print("not there", artefact.short_urn)

## 3 · Publish FMR → .Stat

### Download from the FMR

We ask for the dataflow with `references=all`, so the FMR returns it
together with its DSD, concepts and codelists.

In [8]:
query = StructureQuery(
    artefact_type=StructureType.DATAFLOW,
    agency_id=AGENCY,
    resource_id=f"{PREFIX}_DF_EXR",
    version="1.0",
    detail=StructureDetail.FULL,
    references=StructureReference.ALL,
)
graph = read_sdmx(BytesIO(fmr_service().structure(query)), validate=False)
[s.short_urn for s in graph.structures]

['AgencyScheme=SDMX:AGENCIES(1.0)',
 'Codelist=BIS:PX_SYNC_CL_CURRENCY(1.0)',
 'Codelist=BIS:PX_SYNC_CL_EXR_SUFFIX(1.0)',
 'Codelist=BIS:PX_SYNC_CL_EXR_TYPE(1.0)',
 'Codelist=BIS:PX_SYNC_CL_FREQ(1.0)',
 'Codelist=BIS:PX_SYNC_CL_OBS_STATUS(1.0)',
 'ConceptScheme=BIS:PX_SYNC_ECB_CONCEPTS(1.0)',
 'DataStructure=BIS:PX_SYNC_DSD_EXR(1.0)',
 'Dataflow=BIS:PX_SYNC_DF_EXR(1.0)']

### Submit to .Stat

We drop `SDMX:AGENCIES` so it does not overwrite the one on .Stat, and
submit the rest. If .Stat rejects something, it is reported in
`result`.

In [ ]:
to_stat = [s for s in graph.structures if not isinstance(s, AgencyScheme)]
result = stat_uploader().submit_structure(to_stat, STAT_FORMAT)
print("success:", result.success)
for message in result.messages:
    print(" -", message)

## 4 · Create a difference

Next we put `EXR_OLD` on .Stat only. This is what happens when a
structure is retired in the FMR but stays on .Stat.

In [ ]:
result = stat_uploader().submit_structure(legacy, STAT_FORMAT)
print("success:", result.success, [s.short_urn for s in legacy])

## 5 · Compare FMR and .Stat

We list what each side holds, one structure type at a time (.Stat
rejects all-types queries), and compare their short URNs.

> 🔒 We only compare ids starting with `PX_SYNC_`, because other
> people's structures share the same agency.

In [11]:
TYPES = (
    StructureType.CODELIST,
    StructureType.CONCEPT_SCHEME,
    StructureType.DATA_STRUCTURE,
    StructureType.DATAFLOW,
    StructureType.DATA_CONSTRAINT,
)


def in_scope(artefact) -> bool:
    """Demo scope: only the artefacts this notebook created."""
    return artefact.id.startswith(PREFIX)


def list_urns(svc: RestService) -> set[str]:
    """Short URNs of every in-scope structure of AGENCY on ``svc``."""
    urns = set()
    for stype in TYPES:
        query = StructureQuery(
            artefact_type=stype,
            agency_id=AGENCY,
            detail=StructureDetail.ALL_STUBS,
        )
        try:
            msg = read_sdmx(BytesIO(svc.structure(query)), validate=False)
        except (Invalid, NotFound):
            continue  # no artefacts of this type
        urns |= {s.short_urn for s in msg.structures or [] if in_scope(s)}
    return urns

In [12]:
fmr_urns = list_urns(fmr_service())
sorted(fmr_urns)

['Codelist=BIS:PX_SYNC_CL_CURRENCY(1.0)',
 'Codelist=BIS:PX_SYNC_CL_EXR_SUFFIX(1.0)',
 'Codelist=BIS:PX_SYNC_CL_EXR_TYPE(1.0)',
 'Codelist=BIS:PX_SYNC_CL_FREQ(1.0)',
 'Codelist=BIS:PX_SYNC_CL_OBS_STATUS(1.0)',
 'ConceptScheme=BIS:PX_SYNC_ECB_CONCEPTS(1.0)',
 'DataStructure=BIS:PX_SYNC_DSD_EXR(1.0)',
 'Dataflow=BIS:PX_SYNC_DF_EXR(1.0)']

In [ ]:
stat_urns = list_urns(sdmx_service(f"{NSI}/rest/v2", dotstat_token()))

comparison = pd.DataFrame(
    {
        "short URN": sorted(fmr_urns | stat_urns),
    }
).assign(
    in_FMR=lambda d: d["short URN"].isin(fmr_urns),
    in_Stat=lambda d: d["short URN"].isin(stat_urns),
)
comparison

### What .Stat has that the FMR does not

The difference is what we need to remove. The shared codelists
(`CL_CURRENCY`, …) are still in the FMR, so they are not in it.

In [ ]:
extras = stat_urns - fmr_urns  # present in .Stat, absent in FMR
sorted(extras)

> ⚠️ **Check the list before deleting.** If the FMR listing were
> incomplete, valid structures would show up here too.

## 6 · Deletion order

Structures depend on each other:

```
Dataflow ──▶ DataStructure ──▶ ConceptScheme / Codelist
   ▲
DataConstraint (CR_A_<dataflow>, created by data imports)
```

.Stat rejects deleting a structure that is still in use (409). We can
see it by trying to delete the DSD first:

In [ ]:
wrong_first = sorted(u for u in extras if u.startswith("DataStructure="))
try:
    stat_uploader().delete_structure(wrong_first)
except Invalid as err:
    print("Rejected, as expected:\n", err)

So we sort the list to delete dependents first: constraints,
dataflows, DSDs, then concept schemes and codelists.

In [ ]:
ORDER = {
    "DataConstraint": 0,
    "ContentConstraint": 0,
    "Dataflow": 2,
    "DataStructure": 3,
    "ConceptScheme": 4,
    "Codelist": 5,
}
ordered = sorted(extras, key=lambda u: ORDER.get(u.split("=", 1)[0], 99))
ordered

## 7 · Clean up .Stat and verify

`delete_structure` deletes the list in order and stops at the first
error. If that happens, fix the cause and re-run with what is left.

In [ ]:
try:
    results = stat_uploader().delete_structure(ordered)
    for urn, result in zip(ordered, results):
        print("deleted", urn, "|", "; ".join(result.messages))
except (Invalid, NotFound) as err:
    print("FAILED:", err)

### Verify

We list .Stat again: nothing extra should remain, and everything in
the FMR should still be there.

In [ ]:
stat_after = list_urns(sdmx_service(f"{NSI}/rest/v2", dotstat_token()))
print("still extra on .Stat:", sorted(stat_after - fmr_urns) or "none")
print("FMR structures kept: ", sorted(stat_after & fmr_urns))

## 8 · Recap

| Step | pysdmx |
|---|---|
| Read FMR / .Stat | `RestService` + `StructureQuery` |
| Publish to .Stat | `StatUploader.submit_structure` |
| Compare | set difference of short URNs |
| Clean up .Stat | `StatUploader.delete_structure` |

- The FMR must be complete: it is the reference.
- Check the difference before deleting.
- Delete dependents first (also `CR_A_<dataflow>` after data imports).
- Deletion stops at the first error.
- .Stat tokens expire after 10 minutes.

### Tear-down

Finally we remove `keep` from .Stat so the notebook can be run again.

In [ ]:
try:
    stat_uploader().delete_structure(list(reversed(keep)))
    print("removed from .Stat:", [s.short_urn for s in reversed(keep)])
except (Invalid, NotFound) as err:
    print("FAILED:", err)